# 40. SegFormer 논문 구조 정리와 개발자 관점

이 노트북은 5장의 마지막 정리입니다. SegFormer를 논문 구조와 개발자 관점에서 다시 정리하고, 실제 구현 또는 개선 작업을 할 때 어디를 봐야 하는지 기준을 세웁니다.

이번 노트북의 목표는 다음과 같습니다.

- SegFormer의 핵심 설계 결정을 요약합니다.
- encoder, attention, decoder, loss, metric 관점의 점검 항목을 정리합니다.
- 개발자가 수정하거나 실험하기 좋은 지점을 구분합니다.
- 5장의 전체 흐름을 마무리합니다.

In [ ]:
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.unicode_minus'] = False

## 40-1. SegFormer를 한 문장으로 정리하기

SegFormer는 **hierarchical Transformer encoder(MiT)**와 **lightweight MLP decoder**를 결합한 semantic segmentation 모델입니다.

```text
overlapping patch embedding
  + efficient self-attention
  + multi-scale Transformer features
  + simple MLP decoder
  -> semantic segmentation map
```

## 40-2. 핵심 설계 결정

| 구성 | 설계 의도 | 개발자가 확인할 점 |
|---|---|---|
| Overlapping patch embedding | patch 경계 단절 완화 | kernel, stride, padding, output shape |
| MiT stage structure | multi-scale feature 생성 | stage별 resolution과 channel |
| Efficient self-attention | 고해상도 attention 비용 절감 | reduction ratio와 memory 사용량 |
| Positional encoding 제거 | 입력 해상도 변화에 유연 | 다양한 crop/resize에서 안정성 |
| MLP decoder | 단순하고 빠른 segmentation head | feature align, upsample, concat shape |
| Segmentation loss | pixel-wise supervision | ignore index, class imbalance, label resize |

In [ ]:
components = ['patch embed', 'encoder', 'attention', 'decoder', 'loss/metric']
risk = np.array([3, 4, 4, 3, 5])
debug_cost = np.array([2, 4, 5, 3, 4])

x = np.arange(len(components))
plt.figure(figsize=(8, 3.5))
plt.bar(x - 0.18, risk, width=0.36, label='behavior impact')
plt.bar(x + 0.18, debug_cost, width=0.36, label='debug cost')
plt.xticks(x, components, rotation=20, ha='right')
plt.ylim(0, 5.5)
plt.title('개발자 관점의 점검 우선순위')
plt.legend()
plt.tight_layout()
plt.show()

## 40-3. 구현할 때 자주 생기는 문제

- `B, C, H, W`와 `B, H, W, C`가 섞여 shape mismatch가 발생합니다.
- stage feature의 stride가 기대와 다르면 decoder upsample 비율이 틀어집니다.
- label mask를 image처럼 bilinear resize하면 class id가 깨집니다. mask resize에는 nearest가 필요합니다.
- ignore index가 loss와 metric에서 동일하게 처리되지 않으면 결과가 왜곡됩니다.
- class imbalance가 큰 데이터셋에서는 pixel accuracy가 높아도 mIoU가 낮을 수 있습니다.

## 40-4. 실험 아이디어

SegFormer 개발자라면 다음 축을 기준으로 실험을 설계할 수 있습니다.

| 실험 축 | 바꿔 볼 것 | 관찰할 것 |
|---|---|---|
| Encoder scale | B0, B1, B2 등 모델 크기 | mIoU, latency, memory |
| Reduction ratio | stage별 attention reduction | speed와 small object 성능 |
| Decoder dim | MLP decoder channel | boundary quality와 비용 |
| Loss | CE, Dice, Focal, Lovasz 등 | class imbalance 대응 |
| Augmentation | crop, scale, color jitter | 일반화 성능 |
| Input resolution | train/test crop size | detail 보존과 latency |

## 40-5. Debug checklist

SegFormer 계열 모델을 구현하거나 수정할 때는 아래 순서로 확인하면 좋습니다.

1. 입력 tensor shape와 normalization이 맞는지 확인합니다.
2. encoder stage별 feature shape를 출력합니다.
3. decoder projection 후 channel이 동일한지 확인합니다.
4. upsample 후 모든 feature의 spatial size가 같은지 확인합니다.
5. logits shape가 `B, num_classes, H, W`인지 확인합니다.
6. label mask의 dtype, shape, ignore index를 확인합니다.
7. loss가 감소하는지, overfit small batch가 가능한지 확인합니다.
8. pixel accuracy와 mIoU를 함께 봅니다.

In [ ]:
def check_shapes(image_shape, feature_shapes, logits_shape, label_shape, num_classes):
    print('input image:', image_shape)
    for i, shape in enumerate(feature_shapes, start=1):
        print(f'feature C{i}:', shape)
    print('logits:', logits_shape)
    print('label:', label_shape)
    ok_class = logits_shape[1] == num_classes
    ok_spatial = logits_shape[-2:] == label_shape[-2:]
    print('class dimension ok:', ok_class)
    print('spatial size ok:', ok_spatial)

check_shapes(
    image_shape=(2, 3, 512, 512),
    feature_shapes=[(2, 64, 128, 128), (2, 128, 64, 64), (2, 320, 32, 32), (2, 512, 16, 16)],
    logits_shape=(2, 19, 512, 512),
    label_shape=(2, 512, 512),
    num_classes=19,
)

## 40-6. 5장 전체 흐름 정리

```text
33. SegFormer가 등장한 배경
34. ViT를 segmentation에 그대로 쓰기 어려운 이유
35. MiT encoder 구조
36. Efficient self-attention
37. MLP decoder
38. 전체 forward 흐름
39. 간단 실습과 metric
40. 논문 구조와 개발자 관점 정리
```

이 흐름은 ViT의 token 관점에서 시작해, segmentation에 필요한 dense prediction 구조로 자연스럽게 넘어가도록 설계되어 있습니다.

## 최종 정리

- SegFormer는 ViT 이후의 Transformer 관점을 semantic segmentation에 맞게 재구성한 모델입니다.
- 핵심은 MiT encoder의 multi-scale feature와 가벼운 MLP decoder입니다.
- 개발자 관점에서는 shape, stride, reduction ratio, decoder fusion, label 처리, metric 계산을 특히 꼼꼼히 확인해야 합니다.
- SegFormer를 개선하거나 변형할 때는 encoder 효율, decoder 표현력, loss/augmentation, deployment latency를 분리해서 실험하는 것이 좋습니다.